# Experiment

In [1]:
import pyterrier as pt
import pandas as pd
import os
from pathlib import Path

In [2]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

## Index Creation/Loading

In [27]:
index_dir_path = Path("index/trec_covid_index_default").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "data.properties")

In [28]:
os.makedirs(index_dir, exist_ok=True)

In [29]:
def trec_covid_corpus_generator():
    for doc in dataset.get_corpus_iter():
        title = str(doc.get('title', ''))
        abstract = str(doc.get('abstract', ''))

        yield {
            'docno': doc['docno'],
            'text': title + " " + abstract
        }

In [30]:
if not pt.started():
    pt.init()

C:\Users\zosia\AppData\Local\Temp\ipykernel_36660\3057724015.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


In [31]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. Loading it...")
    index_ref = pt.IndexRef.of(index_properties_file)
else:
    print(f"Index not found at {index_dir}. Building it now (this may take a few minutes)...")
    
    indexer = pt.IterDictIndexer(index_dir, blocks=True, meta={'docno': 50})
    index_ref = indexer.index(trec_covid_corpus_generator())
    print("Index built successfully.")

Index not found at C:\Users\zosia\Desktop\Studies\Information Retrieval\IR_Project\src\index\trec_covid_index_default. Building it now (this may take a few minutes)...


cord19/trec-covid documents:   0%|          | 0/192509 [00:00<?, ?it/s]

12:18:18.092 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
12:19:15.646 [main] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
For MetaIndex, to suppress, set metaindex.compressed.reverse.allow.duplicates=true
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(Indexer.java:377)
12:19:20.908 [main] WA

## Different Query Verbosity Levels

In [32]:
topics_data = []
for query in dataset.irds_ref().queries_iter():
    topics_data.append({
        'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })
df_all_topics = pd.DataFrame(topics_data)

In [33]:
# TITLE Queries
topics_title = df_all_topics[['qid', 'title']].copy()
topics_title = topics_title.rename(columns={'title': 'query'})

# DESCRIPTION Queries
topics_desc = df_all_topics[['qid', 'description']].copy()
topics_desc = topics_desc.rename(columns={'description': 'query'})

# NARRATIVE Queries
topics_narr = df_all_topics[['qid', 'narrative']].copy()
topics_narr = topics_narr.rename(columns={'narrative': 'query'})

## Models

In [34]:
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25")
ql_dir = pt.terrier.Retriever(index_ref, wmodel="DirichletLM")
ql_jm = pt.terrier.Retriever(index_ref, wmodel="Hiemstra_LM")
rm3_pipe = bm25 >> pt.rewrite.RM3(index_ref) >> bm25

In [35]:
models = [bm25, ql_dir, ql_jm, rm3_pipe]
model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3"]

## Experiment

In [36]:
eval_metrics = [
    "ndcg_cut_10",
    "P_5",
    "recip_rank",
    "recall_100",
    "map"
]
query_variants = {
    "Title": topics_title,
    "Description": topics_desc,
    "Narrative": topics_narr
}

In [37]:
for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,0.000784,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,0.000003,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,0.695299,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01




Results for DESCRIPTION Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493




Results for NARRATIVE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629


## Experiment with Dense Retrieval (DPR)

In [14]:
!pip install -q torch torchvision torchaudio transformers
!pip install -q faiss-cpu 
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_dr.git


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import pyterrier_dr as ptdr
import os
from pathlib import Path

dense_index_dir = str(Path("index/trec_covid_dense").resolve())

if os.path.exists(os.path.join(dense_index_dir, "pt_meta.json")):
    print("Found downloaded dense index! Loading...")
    dpr_model = ptdr.SBertBiEncoder('sentence-transformers/all-MiniLM-L6-v2')
    dense_index = ptdr.FlexIndex(dense_index_dir)
    dpr_pipe = dpr_model >> dense_index
    
    print("DPR Pipeline ready!")
else:
    print(f"Could not find the dense index at {dense_index_dir}. Please check where you extracted it!")

Found downloaded dense index! Loading...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR Pipeline ready!


In [39]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR (all-MiniLM)"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,7.844063e-04,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,2.983346e-06,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,6.952988e-01,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,DPR (all-MiniLM),0.075122,0.563015,0.340,0.050606,0.299373,3.0,47.0,2.412138e-09,6.0,...,0.000482,3.0,34.0,1.268344e-08,5.0,43.0,1.140979e-08,6.0,43.0,2.527973e-09




Results for DESCRIPTION Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493
4,DPR (all-MiniLM),0.119940,0.680775,0.524,0.075400,0.448718,6.0,44.0,2.223430e-09,5.0,...,0.004876,9.0,29.0,0.000935,8.0,41.0,1.290954e-06,12.0,38.0,0.000474




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR (all-MiniLM),0.111529,0.542542,0.424,0.069849,0.368355,14.0,36.0,2.405376e-03,9.0,...,0.023023,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000409


# DPR fine-tuning and verbosity (Natalie)

**Goal:** Decide whether one fine-tuned bi-encoder is enough or **separate** models (and indexes) per query verbosity are needed.

**Why one index per fine-tuned checkpoint:** `SBertBiEncoder` shares the same weights for queries and documents. After fine-tuning, rebuilding the `FlexIndex` with that model.

**Protocol:**

1. **Unified model:** Build training pairs with `verbosity="all"` in `dpr_finetune.build_training_pairs` (one pair per relevance judgment × each non-empty topic field). Training once, building one index, evaluating on **Title, Description**, and **Narrative** test queries (same loop as BM25).
2. **Specialist models:** Repeating training with `verbosity="title"` (and similarly `description`, `narrative` only). Each run needs its **own** saved model directory and **its own** dense index.
3. **Compare** MAP / nDCG@10 per test verbosity.

**Training signal:** Using qrels with `label >= 1` (partial + fully relevant) unless metrics group agrees on stricter positives (`min_label=2`).

In [ ]:
# Optional: fine-tune DPR — run in a GPU environment; then rebuild FlexIndex with the same model.
# from pathlib import Path
# import pyterrier_dr as ptdr
#
# from dpr_finetune import build_training_pairs, fit_sentence_transformer
#
# qrels = dataset.get_qrels()
# pairs_all = build_training_pairs(
#     dataset, df_all_topics, qrels, verbosity="all", min_label=1
# )
# out_dir = str(Path("models/dpr_covid_all_minilm").resolve())
# fit_sentence_transformer(
#     pairs_all,
#     "sentence-transformers/all-MiniLM-L6-v2",
#     out_dir,
#     epochs=2,
#     batch_size=16,
# )
#
# # Rebuild dense index (same text field as corpus_generator):
# finetuned = ptdr.SBertBiEncoder(out_dir)
# idx_path = str(Path("index/trec_covid_dense_finetuned_all").resolve())
# dense_new = ptdr.FlexIndex(idx_path)
# (finetuned >> dense_new).index(trec_covid_corpus_generator())
# dpr_pipe_ft = finetuned >> dense_new
#
# # Specialist example: pairs_title = build_training_pairs(..., verbosity="title")
# # Compare pt.Experiment(..., topics=topics_title) for baseline vs unified vs title-only model.

# BioDPR

In [17]:
!pip install -q --upgrade torch torchvision torchaudio

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
import pyterrier_dr as ptdr
import os
from pathlib import Path

biodpr_index_dir = str(Path("index/trec_covid_biodpr_complete").resolve())

if os.path.exists(os.path.join(biodpr_index_dir, "pt_meta.json")):
    print("Found BioDPR index! Loading...")
    
    biodpr_model = ptdr.SBertBiEncoder("pritamdeka/S-PubMedBert-MS-MARCO")
    biodpr_index = ptdr.FlexIndex(biodpr_index_dir)
    biodpr_pipe = biodpr_model >> biodpr_index
    
    print("BioDPR Pipeline ready!")
else:
    print(f"Could not find the index at {biodpr_index_dir}. Please make sure you extracted the folder correctly!")

Found BioDPR index! Loading...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BioDPR Pipeline ready!


In [41]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe, biodpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR", "BioDPR"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,7.844063e-04,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,2.983346e-06,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,6.952988e-01,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,DPR,0.075122,0.563015,0.340,0.050606,0.299373,3.0,47.0,2.412138e-09,6.0,...,0.000482,3.0,34.0,1.268344e-08,5.0,43.0,1.140979e-08,6.0,43.0,2.527973e-09
5,BioDPR,0.100963,0.553502,0.420,0.068487,0.348634,8.0,42.0,3.971513e-07,4.0,...,0.000041,7.0,32.0,2.960906e-06,5.0,44.0,1.265811e-06,9.0,40.0,1.406583e-07




Results for DESCRIPTION Queries:


...

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493
4,DPR,0.119940,0.680775,0.524,0.075400,0.448718,6.0,44.0,2.223430e-09,5.0,...,0.004876,9.0,29.0,0.000935,8.0,41.0,1.290954e-06,12.0,38.0,0.000474
5,BioDPR,0.168095,0.808270,0.652,0.096608,0.597648,14.0,36.0,7.343930e-04,6.0,...,0.204821,18.0,24.0,0.194325,15.0,32.0,3.710228e-02,24.0,26.0,0.544790




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR,0.111529,0.542542,0.424,0.069849,0.368355,14.0,36.0,2.405376e-03,9.0,...,0.023023,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000409
5,BioDPR,0.140223,0.786781,0.600,0.089901,0.551742,20.0,30.0,2.476971e-01,12.0,...,0.245488,17.0,18.0,0.878472,21.0,28.0,0.903510,25.0,23.0,0.409129


# Hybrid Approach

In [42]:
import pyterrier as pt

def apply_rrf(res, k=60):
    res = res.copy()
    res['score'] = 1.0 / (k + res['rank'] + 1)
    return res

rrf_transformer = pt.apply.generic(apply_rrf)

bm25_rrf = bm25 >> rrf_transformer
biodpr_rrf = biodpr_pipe >> rrf_transformer

hybrid_pipe = bm25_rrf + biodpr_rrf

all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe, biodpr_pipe, hybrid_pipe]
all_model_names = [
    "BM25", 
    "QL (Dir)", 
    "QL (JM)", 
    "BM25 + RM3", 
    "DPR", 
    "BioDPR", 
    "Hybrid (BM25 + BioDPR RRF)"
]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


c:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #6: Hybrid (BM25 + BioDPR RRF) (<pyterrier._ops.Sum object at 0x000001F99FE90740>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

...

c:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #6: Hybrid (BM25 + BioDPR RRF) (<pyterrier._ops.Sum object at 0x000001F99FE90740>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR,0.111529,0.542542,0.424,0.069849,0.368355,14.0,36.0,2.405376e-03,9.0,...,0.023023,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000409
5,BioDPR,0.140223,0.786781,0.600,0.089901,0.551742,20.0,30.0,2.476971e-01,12.0,...,0.245488,17.0,18.0,0.878472,21.0,28.0,0.903510,25.0,23.0,0.409129
6,Hybrid (BM25 + BioDPR RRF),0.207661,0.804306,0.712,0.104749,0.640086,41.0,9.0,1.232110e-07,13.0,...,0.143505,19.0,10.0,0.018952,36.0,12.0,0.000331,34.0,13.0,0.000133
